# Stage 3 — SMA crossover versus buy-and-hold

**Self-directed academic project.** This notebook studies one rule; it is not a live trading recommendation.

A moving average uses past adjusted closing prices. A fast average above a slow average creates a long signal. The signal is computed at a close, filled at a later close, and only earns returns after that assumed fill. This deliberate delay avoids pretending that the close was known before trading at that same close. Both strategies pay configured transaction costs and slippage; buy-and-hold pays to enter and to liquidate at the end.

The fixed present-day SPY benchmark is not point-in-time data. Adjusted prices can change, and this single benchmark cannot demonstrate that a strategy generalises.

In [ ]:
from portfolio_research.backtest import run_sma_vs_buy_and_hold
from portfolio_research.market_data import load_prices
from portfolio_research.metrics import calculate_metrics

data = load_prices(["SPY"], start="2005-01-03", end="2026-01-01")
spy = data.prices["SPY"]
print(f"Source: {data.source}; observations: {data.quality.observation_count}")
print(f"Missing values: {data.quality.missing_by_symbol}")

In [ ]:
comparison = run_sma_vs_buy_and_hold(spy)
sma_metrics = calculate_metrics(
    comparison.sma.net_returns, comparison.buy_and_hold.net_returns
)
benchmark_metrics = calculate_metrics(
    comparison.buy_and_hold.net_returns,
    comparison.buy_and_hold.net_returns,
)
print("SMA metrics:", sma_metrics)
print("Buy-and-hold metrics:", benchmark_metrics)

In [ ]:
import pandas as pd

wealth = pd.DataFrame(
    {
        "SMA, net of costs": (1 + comparison.sma.net_returns).cumprod(),
        "Buy-and-hold, net of costs": (
            1 + comparison.buy_and_hold.net_returns
        ).cumprod(),
    }
)
display(wealth.iloc[::252].tail(10))
display(
    pd.DataFrame(
        {
            "SMA position": comparison.sma.positions,
            "SMA trading cost": comparison.sma.transaction_costs,
            "Buy-and-hold trading cost": comparison.buy_and_hold.transaction_costs,
        }
    ).tail()
)

## Interpret before celebrating

Compare net results, not gross returns. Ask whether the strategy's drawdown and risk-adjusted results justify its time out of the market. A single chosen SMA pair and one ETF do not constitute out-of-sample validation; parameter search can overfit, and adjusted daily bars do not model intraday fills, market impact, taxes, or all real-world trading frictions.